# Проверка: пересечение H2O и CRM по ФП

**Цель:** выяснить, какая доля записей H2O (с `ref_book_fp_id`) присутствует в CRM,
и нужны ли нам записи H2O, которых нет в CRM.

**Связка:** `h2o.crm_fp_id` ↔ `crm.ROW_ID`  
**Фильтр CRM:** `VAL = 'H2.0'` (только записи из источника H2O)

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np

# --- H2O ---
df_h2o = pd.read_excel("../sources/data_h20.xlsx", dtype=str)
df_h2o.columns = df_h2o.columns.str.strip()
print(f"H2O: {len(df_h2o):,} строк")

# --- CRM (полная выгрузка, без фильтров) ---
df_crm = pd.read_csv("../sources/data_crm.csv", sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
print(f"CRM: {len(df_crm):,} строк")

# Подготовка ключей H2O (один раз)
h2o_ids = df_h2o["crm_fp_id"].dropna().astype(str).str.strip()
h2o_ids = set(h2o_ids[h2o_ids != ""])
print(f"\nУникальных crm_fp_id в H2O (непустых): {len(h2o_ids):,}")

## 2. Сравнение БЕЗ фильтров (полный CRM vs H2O)

In [ ]:
crm_all_keys = set(
    df_crm["ROW_ID"].dropna().astype(str).str.strip()
)
crm_all_keys.discard("")

both_no_filter = h2o_ids & crm_all_keys
only_h2o_no_filter = h2o_ids - crm_all_keys

print(f"CRM ROW_ID (весь CRM, без фильтров):  {len(crm_all_keys):,}")
print(f"H2O crm_fp_id:                         {len(h2o_ids):,}")
print()
print(f"Найдены и в H2O, и в CRM:              {len(both_no_filter):,}")
print(f"Есть в H2O, но НЕТ в CRM:              {len(only_h2o_no_filter):,}")

# Сколько строк H2O это затрагивает
n_rows_found = df_h2o["crm_fp_id"].astype(str).str.strip().isin(both_no_filter).sum()
n_rows_missing = df_h2o["crm_fp_id"].astype(str).str.strip().isin(only_h2o_no_filter).sum()
print(f"\nВ строках H2O:")
print(f"  Найдены в CRM:   {n_rows_found:,}")
print(f"  Нет в CRM:       {n_rows_missing:,}")

## 3. Сравнение С фильтрами (VAL = 'H2.0', дата ≥ 2023-02-01)

In [ ]:
crm_filtered = df_crm[
    (df_crm["VAL"].str.strip() == "H2.0") &
    (df_crm["IDENTIFICATION_DATE"] >= "2023-02-01")
].copy()
print(f"CRM после фильтров (VAL='H2.0' + дата ≥ 2023-02-01): {len(crm_filtered):,} строк")

crm_filtered_keys = set(
    crm_filtered["ROW_ID"].dropna().astype(str).str.strip()
)
crm_filtered_keys.discard("")

both_filtered = h2o_ids & crm_filtered_keys
only_h2o_filtered = h2o_ids - crm_filtered_keys

print(f"CRM ROW_ID (с фильтрами):              {len(crm_filtered_keys):,}")
print(f"H2O crm_fp_id:                         {len(h2o_ids):,}")
print()
print(f"Найдены и в H2O, и в CRM:              {len(both_filtered):,}")
print(f"Есть в H2O, но НЕТ в CRM:              {len(only_h2o_filtered):,}")

n_rows_found_f = df_h2o["crm_fp_id"].astype(str).str.strip().isin(both_filtered).sum()
n_rows_missing_f = df_h2o["crm_fp_id"].astype(str).str.strip().isin(only_h2o_filtered).sum()
print(f"\nВ строках H2O:")
print(f"  Найдены в CRM:   {n_rows_found_f:,}")
print(f"  Нет в CRM:       {n_rows_missing_f:,}")

## 4. Сводка: влияние фильтров на пересечение

In [ ]:
summary = pd.DataFrame({
    "Сценарий": ["Без фильтров (полный CRM)", "С фильтрами (VAL='H2.0' + дата ≥ 2023-02-01)"],
    "ROW_ID в CRM": [len(crm_all_keys), len(crm_filtered_keys)],
    "Найдены в обоих": [len(both_no_filter), len(both_filtered)],
    "Только в H2O": [len(only_h2o_no_filter), len(only_h2o_filtered)],
})
display(summary.style.hide(axis="index"))

lost_by_filters = len(only_h2o_filtered) - len(only_h2o_no_filter)
print(f"\nФильтры добавили {lost_by_filters:,} «потерянных» записей H2O")
print(f"  → {len(only_h2o_no_filter):,} отсутствуют в CRM изначально (нет ROW_ID вообще)")
print(f"  → {lost_by_filters:,} отсечены фильтрами (VAL ≠ 'H2.0' или дата < 2023-02-01)")

## 5. Анализ записей H2O, которых нет в CRM

Кто эти «лишние» записи? Есть ли у них `ref_book_fp_id`?

In [ ]:
# Записи H2O, чей crm_fp_id НЕ нашёлся в CRM (с фильтрами)
h2o_not_in_crm = df_h2o[
    df_h2o["crm_fp_id"].astype(str).str.strip().isin(only_h2o_filtered)
].copy()

# Записи H2O вообще без crm_fp_id (пустые)
h2o_no_key = df_h2o[
    df_h2o["crm_fp_id"].isna() |
    (df_h2o["crm_fp_id"].astype(str).str.strip() == "")
]

# Записи H2O, которые нашлись в CRM (с фильтрами)
h2o_in_crm = df_h2o[
    df_h2o["crm_fp_id"].astype(str).str.strip().isin(both_filtered)
]

print(f"H2O записей: всего {len(df_h2o):,}")
print(f"  — нашлись в CRM (с фильтрами):       {len(h2o_in_crm):,}")
print(f"  — crm_fp_id есть, но нет в CRM:      {len(h2o_not_in_crm):,}")
print(f"  — crm_fp_id пустой:                   {len(h2o_no_key):,}")
print(f"  — итого «не в CRM»:                   {len(h2o_not_in_crm) + len(h2o_no_key):,}")

In [ ]:
# Есть ли ref_book_fp_id у «лишних» записей?
print("ref_book_fp_id у записей H2O, которых нет в CRM:")
not_in_crm_all = pd.concat([h2o_not_in_crm, h2o_no_key])
n_with_fp = not_in_crm_all["ref_book_fp_id"].notna().sum()
n_without_fp = not_in_crm_all["ref_book_fp_id"].isna().sum()
print(f"  С ref_book_fp_id:    {n_with_fp:,}")
print(f"  Без ref_book_fp_id:  {n_without_fp:,}")

if n_with_fp > 0:
    print(f"\nТипы ФП (ref_book_fp_id) у 'лишних':")
    print(not_in_crm_all["ref_book_fp_id"].value_counts().head(20).to_string())

## 5.1. Диагностика: почему 22 970 строк H2O не нашлись в CRM?

In [ ]:
# Записи H2O, которых нет в CRM вообще (даже без фильтров)
truly_missing = df_h2o[
    df_h2o["crm_fp_id"].astype(str).str.strip().isin(only_h2o_no_filter)
]
print(f"Записей H2O, которых нет в CRM ВООБЩЕ (без фильтров): {len(truly_missing):,}")
if len(truly_missing) > 0:
    print("\nПримеры:")
    display(truly_missing.head(10))

# Записи H2O, которые есть в CRM, но отсечены фильтрами
lost_ids = only_h2o_filtered - only_h2o_no_filter
lost_by_filter = df_h2o[
    df_h2o["crm_fp_id"].astype(str).str.strip().isin(lost_ids)
]
print(f"\n{'='*70}")
print(f"Записей H2O, отсечённых фильтрами (VAL ≠ 'H2.0' или дата < 2023-02-01): {len(lost_by_filter):,}")

if len(lost_ids) > 0:
    sample_lost = list(lost_ids)[:500]
    crm_lost = df_crm[df_crm["ROW_ID"].astype(str).str.strip().isin(sample_lost)]
    print(f"\nИз {len(sample_lost)} проверенных — в каком VAL они лежат в CRM:")
    print(crm_lost["VAL"].value_counts(dropna=False).to_string())
    print(f"\nДиапазон IDENTIFICATION_DATE:")
    print(f"  min: {crm_lost['IDENTIFICATION_DATE'].min()}")
    print(f"  max: {crm_lost['IDENTIFICATION_DATE'].max()}")
    print(f"\nПримеры:")
    display(crm_lost[["ROW_ID", "VAL", "IDENTIFICATION_DATE", "X_INN"]].head(10))

## 6. Анализ пропусков ref_book_fp_id: связь с наличием в CRM

In [ ]:
df_h2o["_in_crm"] = df_h2o["crm_fp_id"].astype(str).str.strip().isin(both_filtered)
df_h2o["_has_fp_type"] = df_h2o["ref_book_fp_id"].notna()

cross = pd.crosstab(
    df_h2o["_in_crm"].map({True: "Есть в CRM", False: "Нет в CRM"}),
    df_h2o["_has_fp_type"].map({True: "ref_book_fp_id есть", False: "ref_book_fp_id пусто"}),
    margins=True,
)
print("Кросс-таблица: наличие в CRM × наличие ref_book_fp_id")
display(cross)

df_h2o.drop(columns=["_in_crm", "_has_fp_type"], inplace=True)

## 7. Вывод

Заполните после просмотра результатов:

- Записей H2O, которых нет в CRM: **___**
- Из них с ref_book_fp_id: **___**
- **Решение:** если «лишние» записи H2O без ref_book_fp_id и/или без crm_fp_id — они не нужны для анализа, можно безопасно отбросить.